In [58]:
import requests
import json

import sys
import os
sys.path.append(os.path.abspath('..'))
from shared import generate
import time




### Figurative

In [181]:
def ask_gpt(system, prompt, model):
  return generate(model = model,
        system = system,
        query = prompt,
        temperature=0,
        lastk=0,
        session_id='new',
        rag_usage = True,
        rag_threshold = 0,
        rag_k = 0)


#### Extract Idioms

In [152]:
with open("figurative.json", "r") as f:
    data = json.load(f)

In [141]:
system = '''Your task is to identify and extract figurative phrases.'''
prompt = '''Identify and return the figurative phrase in the sentence below. Return it as a self-contained phrase that can be read as a whole. It must not be longer than 5 words. If required, rephrase it to shorten it to upto 5 words. Here is the sentence:\n\n{sentence}'''
for index in range(len(data)):
    ret = ask_gpt(system, prompt.format(sentence = data[index]['Speaker 1']), 'gpt-4o')['response']
    data[index]['fig']=ret
    


too many irons in the fire
pulled a rabbit out of his hat
burning the candle at both ends
let the cat out
get our ducks in a row
hit the nail on the head
biting off more than he can chew
took the wind out
juggling too many tasks
apple of his eye
strike while the iron is hot
keeping our cards close
all his eggs in one basket
a mountain out of a molehill
lit a fire under
the glue that holds


In [142]:
with open("figurative.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

#### Opaqueness

In [177]:
with open("figurative.json", "r") as f:
    data = json.load(f)

In [182]:
system = '''You are an expert in figurative language and semantics.'''

prompt = '''Your task is to classify idioms based on their transparency (compositionality), i.e., how much the figurative meaning can be inferred from the literal meanings of the words.

Use the following scale strictly:
0 = Opaque -- meaning cannot be inferred from the words (e.g., "kick the bucket")
1 = Semi-transparent -- partial clues from words but not fully inferable (e.g., "hold your tongue")
2 = Transparent -- meaning is easily inferred from the words (e.g., "see the light")

Here is the idiom:\n\n{phrase}

Return output strictly in JSON with:

  "score": 0 | 1 | 2,
  "label": "opaque" | "semi-transparent" | "transparent",
  "reason": "one short sentence explaining why"

'''

for index in range(len(data)):
  fig = data[index]['fig']
  ret = json.loads(ask_gpt(system, prompt.format(phrase = fig), 'gpt-4o')['response'])
  data[index]['label']=ret['label']
  data[index]['reason']=ret['reason']
  print(ret)

{'score': 1, 'label': 'semi-transparent', 'reason': 'The idea of something becoming thin suggests weakening, but the specific sense of patience or tolerance diminishing is not fully predictable from the words alone.'}
{'score': 2, 'label': 'transparent', 'reason': 'It clearly means the intelligent planner or mastermind of an activity, which follows directly from the literal idea of a brain controlling an operation.'}
{'score': 1, 'label': 'semi-transparent', 'reason': 'The idea of multiple irons in a fire suggests several tasks or projects underway, but the specific figurative meaning is not fully clear from the words alone.'}
{'score': 2, 'label': 'transparent', 'reason': 'The idea of the heart as character and gold as great value makes the positive figurative meaning easy to infer.'}
{'score': 1, 'label': 'semi-transparent', 'reason': 'The idea of having wings suggests speed or ability to fly away, which hints at things (like time or money) disappearing quickly but does not fully spe

In [183]:
with open("figurative.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)


#### Retrieving NGram Counts

In [85]:


BASE = "https://api.ngrams.dev"
CORPUS = "eng"

def two_total_count_2000_2022(phrase: str) -> int:
    # Search is case-insensitive by default.
    # Use rq so the phrase is treated literally, not as query syntax.
    search_url = f"{BASE}/{CORPUS}/search"
    params = {
        "query": phrase,
        "flags": "rq",
        "limit": 100,
    }

    total = 0
    next_token = None

    while True:
        p = dict(params)
        if next_token:
            p["start"] = next_token

        r = requests.get(search_url, params=p, timeout=30)
        r.raise_for_status()
        search_res = r.json()

        for ng in search_res.get("ngrams", []):
            ng_id = ng["id"]
            rr = requests.get(f"{BASE}/{CORPUS}/{ng_id}", timeout=30)
            rr.raise_for_status()
            full = rr.json()

            for stat in full.get("stats", []):
                year = stat["year"]
                if 2000 <= year <= 2022:
                    total += stat["absMatchCount"]

        next_token = search_res.get("nextPageToken")
        if not next_token:
            break

    return total

In [ ]:
#Check if exists. 5> do not exist.

phrase = "lite fire under the team"

r = requests.get(
    f"{BASE}/{CORPUS}/search",
    params={"query": phrase, "flags": "rq", "limit": 100},
    timeout=30
)
print(r.url)
print(r.status_code)
print(r.json())

In [145]:
print(two_total_count_2000_2022("bite more than can chew"))

0


## Blunt

In [14]:
with open("blunt.json", "r") as f:
    data = json.load(f)

In [33]:
count = 0
for index in range(len(data)):
    #print(data[index]['Speaker 1'][0])
    if data[index]['Speaker 1'][0] == 'I':
        data[index]["label"] = "self"
    else:
        data[index]["label"] = "nonself"


0


In [35]:
with open("blunt.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

[{'ngram': 'heart of gold', 'parent': '', 'type': 'NGRAM', 'timeseries': [8.74022916264039e-08, 9.042380924029204e-08, 9.362952132126641e-08, 9.590882196367342e-08, 9.991847654029179e-08, 1.0535269317253032e-07, 1.118633323114539e-07, 1.1804030058166453e-07, 1.2668736510639584e-07, 1.334994692570035e-07, 1.3910412580539093e-07, 1.5048343422157423e-07, 1.594409561838412e-07, 1.6343639280031702e-07, 1.702731477085503e-07, 1.7436325906926088e-07, 1.7720129059333495e-07, 1.8193196874941955e-07, 1.8057010322536371e-07, 1.7958694087383265e-07]}]


In [7]:
print(sum(data[0]['timeseries']))

2.739117700351861e-06
